In [ ]:
from datasets import load_dataset, concatenate_datasets

print("1. Loading data")
dataset = load_dataset("rohan2810/amazon-movies-meta-reviews-merged", split="train")

print("2. Keeping only required columns")
dataset = dataset.select_columns(['rating', 'cleaned_text'])

print("3. Removing neutral reviews and converting other")
dataset = dataset.filter(lambda x: x['rating'] != 3)
def convert_to_binary(example):
    example['label'] = 1 if example['rating'] > 3 else 0
    return example

dataset = dataset.map(convert_to_binary)

print("4. Splitting into positive and negative")
positives = dataset.filter(lambda x: x['label'] == 1)
negatives = dataset.filter(lambda x: x['label'] == 0)

print("5. Selecting 30,000 random samples of each class...")
pos_sampled = positives.shuffle(seed=42).select(range(30000))
neg_sampled = negatives.shuffle(seed=42).select(range(30000))

print("6. Assembling the final dataset")
balanced_dataset = concatenate_datasets([pos_sampled, neg_sampled])
final_dataset = balanced_dataset.shuffle(seed=42)

print("7. Done")
print(final_dataset[0])

print("last step. spliting data in train/test dataset's")
split_dataset = final_dataset.train_test_split(test_size=0.1, seed=42)
print(split_dataset)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from collections import OrderedDict
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased") #BERT tokenizer

vocab_size = tokenizer.vocab_size # ------------------------------------------------------------------------------------------------------------------------------------take from here

def tokenize_function(examples):
    return tokenizer(
        examples["cleaned_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

split_dataset = final_dataset.train_test_split(test_size=0.1, seed=42)

print("3. Processing texts through the tokenizer")
tokenized_datasets = split_dataset.map(tokenize_function, batched=True)

print("4. Packaging into PyTorch format")
tokenized_datasets = tokenized_datasets.remove_columns(["cleaned_text", "rating"])
tokenized_datasets.set_format("torch")

BATCH_SIZE = 32


# 1. Training DataLoader (shuffle for training)
train_dataloader = DataLoader(
    tokenized_datasets['train'],
    batch_size=BATCH_SIZE,
    shuffle=True
)

# 2. Test DataLoader (no need to shuffle for testing)
test_dataloader = DataLoader(
    tokenized_datasets['test'],
    batch_size=BATCH_SIZE,
    shuffle=False
)


for batch in train_dataloader:
    print("\nBatch contents (training):")
    print("input_ids shape:", batch['input_ids'].shape)
    print("attention_mask shape:", batch['attention_mask'].shape)
    print("label shape:", batch['label'].shape)
    break

In [ ]:
class MultiHeadAttention(nn.Module):
    # Removed masked=False from init as the mask is now passed in forward
    def __init__(self, input_size, head_size, num_heads, out_size, query_input_size=None):
        super(MultiHeadAttention, self).__init__()

        self.input_size = input_size
        self.head_size = head_size
        self.num_heads = num_heads
        self.out_size = out_size
        self.query_input_size = self.input_size if query_input_size is None else query_input_size

        self.W_Q = nn.Linear(self.query_input_size, self.num_heads * self.head_size, bias=False)
        self.W_K = nn.Linear(self.input_size, self.num_heads * self.head_size, bias=False)
        self.W_V = nn.Linear(self.input_size, self.num_heads * self.head_size, bias=False)

        self.out = nn.Linear(self.head_size * self.num_heads, self.out_size)

        self.dropout = nn.Dropout(p=0.1)

    def forward(self, query, key, value, padding_mask=None):
        batch_size = key.size(0)
        emb_len = key.size(1)
        query_emb_len = query.size(1)

        q = self.W_Q(query)
        k = self.W_K(key)
        v = self.W_V(value)

        q = q.view(batch_size, query_emb_len, self.num_heads, self.head_size).transpose(1,2)
        k = k.view(batch_size, emb_len, self.num_heads, self.head_size).transpose(1,2)
        v = v.view(batch_size, emb_len, self.num_heads, self.head_size).transpose(1,2)

        k_T = k.transpose(2, 3)

        relevance = q @ k_T / math.sqrt(self.head_size)
        # relevance size: [batch_size, num_heads, query_emb_len, emb_len]

        if padding_mask is not None:
            # Tokenizer mask has size [batch_size, emb_len].
            # We need to transform it into [batch_size, 1, 1, emb_len] to apply it to relevance.

            # 1. Expand dimensions: [batch_size, 1, 1, emb_len]
            extended_mask = padding_mask.unsqueeze(1).unsqueeze(2)

            # 2. 0 -> -infinity to ensure 0 after softmax
            relevance = relevance.masked_fill(extended_mask == 0, float('-inf'))

        relevance = F.softmax(relevance, dim=-1)

        relevance = self.dropout(relevance)

        heads = relevance @ v
        heads = heads.transpose(1, 2)
        concat = heads.reshape(batch_size, query_emb_len, self.head_size * self.num_heads)
        out = self.out(concat)

        return out

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_emb_len, d_model):
        super(PositionalEncoding, self).__init__()

        self.max_emb_len = max_emb_len
        self.d_model = d_model

        pos = torch.arange(max_emb_len)[:, None] # [[0], [1], [2], [3], [4], ...[max_emb_len]]]
        i = torch.arange(d_model)[None, :] # i = [[0, 1, 2, 3, 4, ..., d_model]]

        pe = torch.zeros(self.max_emb_len, self.d_model)
        # max_emb_len, d_model

        sin = torch.sin(pos / (10000 ** (i[:, ::2] / self.d_model)))
        # max_emb_len, d_model // 2

        cos = torch.cos(pos / (10000 ** (i[:, 1::2] / self.d_model)))
        # max_emb_len, d_model // 2

        pe[:, ::2] = sin
        pe[:, 1::2] = cos

        pe = pe.unsqueeze(0)
        # 1, max_emb_len, d_model

        self.register_buffer('pe', pe)
        self.dropout = nn.Dropout(p=0.1)

    def forward(self, emb):
        # batch_size, emb_len, input_size

        emb_len = emb.size(1)

        emb = emb + self.pe[:, :emb_len]
        # batch_size, emb_len, input_size

        emb = self.dropout(emb)

        return emb

In [ ]:
class EncoderBlock(nn.Module):
    # keeping query_input_size for decoder support
    def __init__(self, input_size, head_size, num_heads, out_size, ff_hidden_size, query_input_size=None):
        super(EncoderBlock, self).__init__()

        self.input_size = input_size
        self.head_size = head_size
        self.num_heads = num_heads
        self.out_size = out_size
        self.query_input_size = input_size if query_input_size is None else query_input_size

        # for feed forward
        self.ff_hidden_size = ff_hidden_size

        self.attention = MultiHeadAttention(
            input_size=self.input_size,
            head_size=self.head_size,
            num_heads=self.num_heads,
            out_size=self.out_size,
            query_input_size=self.query_input_size
        )

        if self.query_input_size != self.out_size:
            self.adapt = nn.Linear(self.query_input_size, self.out_size)
        else:
            self.adapt = nn.Identity() # returns input without changes

        self.norm_1 = nn.LayerNorm(self.out_size)

        self.feed_forward = nn.Sequential(OrderedDict([
            ("Linear_1", nn.Linear(self.out_size, self.ff_hidden_size)),
            ("Activation", nn.ReLU()),
            ("Dropout", nn.Dropout(p=0.1)),
            ("Linear_2", nn.Linear(self.ff_hidden_size, self.out_size)),
        ]))

        self.norm_2 = nn.LayerNorm(self.out_size)
        self.dropout = nn.Dropout(p=0.1)

    def forward(self, query, key, value, padding_mask=None):
        # batch_size, seq_len, in_size

        # Self Multi-Head Attention
        attention_out = self.attention(query, key, value, padding_mask=padding_mask)
        # batch_size, seq_len, out_size

        attention_out = self.dropout(attention_out)

        add_1_out = attention_out + self.adapt(query)
        norm_1_out = self.norm_1(add_1_out)
        # batch_size, seq_len, out_size

        # Feed Forward
        feed_forward_out = self.feed_forward(norm_1_out)
        # batch_size, seq_len, out_size

        feed_forward_out = self.dropout(feed_forward_out)

        # Add + Norm
        add_out = feed_forward_out + norm_1_out
        norm_2_out = self.norm_2(add_out)

        return norm_2_out

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, N, max_seq_len, num_embeddings, emb_size, att_out_size, att_head_size, num_heads, ff_hidden_size):
        super(TransformerEncoder, self).__init__()

        self.N = N
        self.max_seq_len = max_seq_len

        # for embedding_layer
        self.num_embeddings = num_embeddings # number of unique indices (vocab_size)
        self.emb_size = emb_size # embedding vector dimension, d_model

        self.att_out_size = att_out_size
        self.att_head_size = att_head_size
        self.num_heads = num_heads

        self.ff_hidden_size = ff_hidden_size

        self.embedding_layer = nn.Embedding(
            num_embeddings=self.num_embeddings,
            embedding_dim=self.emb_size
        )
        self.positional_encoder = PositionalEncoding(
            max_emb_len=self.max_seq_len,
            d_model=self.emb_size
        )

        self.encoder_blocks = nn.ModuleDict({
            f"encoder_block_{i}": EncoderBlock(
                input_size=self.emb_size if i==0 else self.att_out_size,
                head_size=self.att_head_size,
                num_heads=self.num_heads,
                out_size=self.att_out_size,
                ff_hidden_size=self.ff_hidden_size,
            ) for i in range(self.N)
        })

    def forward(self, encoder_input, padding_mask=None):
        # batch_size, seq_len

        encoder_emb = self.embedding_layer(encoder_input)
        # batch_size, seq_len, emb_size

        out = self.positional_encoder(encoder_emb)
        # batch_size, seq_len, emb_size

        for block in self.encoder_blocks.values():
            out = block(out, out, out, padding_mask=padding_mask)
        # batch_size, seq_len, att_out_size

        return out

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, encoder, hidden_size, num_classes=2):
        super(SentimentClassifier, self).__init__()
        # Accepts a pre-assembled Transformer-encoder
        self.encoder = encoder

        # Final linear layer producing 2 logits
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, padding_mask):
        # encoder_out shape: [batch_size, seq_len, hidden_size]
        encoder_out = self.encoder(input_ids, padding_mask=padding_mask)

        # Expand mask from [batch_size, seq_len] to [batch_size, seq_len, 1]
        mask_expanded = padding_mask.unsqueeze(-1).float()

        # Padding vectors are zeroed out and no longer carry noise.
        masked_embeddings = encoder_out * mask_expanded

        # Sum vectors of all words in the sentence
        summed = torch.sum(masked_embeddings, dim=1)

        # clamp(min=1e-9) protects against division by zero
        counts = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)

        # pooled_out shape: [batch_size, hidden_size]
        pooled_out = summed / counts

        # logits shape: [batch_size, 2]
        logits = self.classifier(pooled_out)

        return logits

In [ ]:
# 1. Set model hyperparameters (values can be tuned)
VOCAB_SIZE = tokenizer.vocab_size # vocabulary size for bert-base-uncased 30522
MAX_SEQ_LEN = 128  # sequence length set in tokenizer
EMB_SIZE = 256     # embedding size
NUM_HEADS = 8      # number of attention heads
FF_HIDDEN_SIZE = 512 # size of hidden layer in FeedForward
NUM_BLOCKS = 3     # number of Encoder blocks

# 2. Create the Encoder
my_encoder = TransformerEncoder(
    N=NUM_BLOCKS,
    max_seq_len=MAX_SEQ_LEN,
    num_embeddings=VOCAB_SIZE,
    emb_size=EMB_SIZE,
    att_out_size=EMB_SIZE,
    att_head_size=EMB_SIZE // NUM_HEADS, # Head size = emb_size / num_heads
    num_heads=NUM_HEADS,
    ff_hidden_size=FF_HIDDEN_SIZE
)

# 3. Wrap the Encoder in our Classifier
model = SentimentClassifier(
    encoder=my_encoder,
    hidden_size=EMB_SIZE,
    num_classes=2
)

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model)

In [ ]:
import torch.optim as optim

# Loss function for classification
criterion = nn.CrossEntropyLoss()

# lr (learning rate) = 1e-4 or 5e-5 is standard for small transformers
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# Number of epochs
EPOCHS = 3

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Training settings
EPOCHS = 3
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}")

    model.train() # Enable Dropout and weight updates
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for step, batch in enumerate(train_dataloader):
        # 1. Move data to GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        # 2. Clear old gradients
        optimizer.zero_grad()

        # 3. Predict and calculate loss
        logits = model(input_ids, padding_mask=attention_mask)
        loss = criterion(logits, labels)

        # 4. Update weights
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Gradient clipping
        optimizer.step()

        train_loss += loss.item()
        predictions = torch.argmax(logits, dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

        # Print progress every 100 batches
        if (step + 1) % 100 == 0:
            print(f"  Step {step+1:4d} | Loss: {train_loss/(step+1):.4f} | Accuracy: {(train_correct/train_total)*100:.2f}%")

    # Test
    model.eval() # Disable Dropout, set to evaluation mode
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # Make predictions
            logits = model(input_ids, padding_mask=attention_mask)
            predictions = torch.argmax(logits, dim=1)

            # Count correct answers
            test_correct += (predictions == labels).sum().item()
            test_total += labels.size(0)

    # Summary
    epoch_train_loss = train_loss / len(train_dataloader)
    epoch_train_acc = (train_correct / train_total) * 100
    epoch_test_acc = (test_correct / test_total) * 100

    print(f"\nEPOCH {epoch + 1} SUMMARY:")
    print(f"Training: Loss: {epoch_train_loss:.4f} | Accuracy: {epoch_train_acc:.2f}%")
    print(f"Test: {epoch_test_acc:.2f}")

In [ ]:
def predict_review(text, model, tokenizer, device, max_len=128):
    model.eval() # Switch to evaluation mode (no Dropout)

    # 1. Tokenize text (same as dataset processing)
    encoded = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors='pt' # Request PyTorch tensors
    )

    # Move to GPU
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    # 2. Make prediction
    with torch.no_grad():
        logits = model(input_ids, padding_mask=attention_mask)
        prediction = torch.argmax(logits, dim=1).item()

        # Calculate model confidence from 0 to 100%
        probabilities = F.softmax(logits, dim=1)
        confidence = probabilities[0][prediction].item() * 100

    # 3. Output result
    sentiment = "POSITIVE" if prediction == 1 else "NEGATIVE"
    print(f"Review: '{text}'")
    print(f"Verdict: {sentiment} (Confidence: {confidence:.1f}%)\n")

predict_review("This movie is absolutely fantastic, a true masterpiece!", model, tokenizer, device)
predict_review("What a waste of time. The acting was terrible and the plot made no sense.", model, tokenizer, device)
predict_review("It is middle movie it is not good and not bad", model, tokenizer, device)